# `pt.pivot` / `df.pt.pivot` — Excel-Style 2D Pivot Tables

`pt.pivot()` summarizes and aggregates tabular data across dimensions using the intuitive **RCVA** grammar:
- **`r`** (Rows): Row grouping dimensions (*optional*)
- **`c`** (Cols): Column header dimensions (*optional*)
- **`v`** (Values): Value / measure column(s) to aggregate (**required**)
- **`a`** (Aggregation): Aggregation function (`sum`, `mean`, `median`, `min`, `max`, `count`, `std`, `n`; default: `sum`)

### Core Guarantees:
1. **Always Flat 1D Output**: Never produces MultiIndex rows or tuples in column headers. Column names are joined with `_` (e.g. `Sales_2023`).
2. **Always Clean Reset Index**: Returns a standard `RangeIndex(0, 1, 2, ...)` every time.
3. **Automatic Subsetting**: Only references `r`, `c`, and `v`—no need to pre-select columns.
4. **No Silent Data Loss**: `dropna=False` by default, retaining missing categories.

In [1]:
import pytae as pt
import pandas as pd

penguins = pt.sample("penguins")
flights = pt.sample("flights")
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


## 1. Classic 2D Contingency Grid (`r=`, `c=`, `v=`)

Summarize average body mass across islands (rows) and species (columns):

In [2]:
penguins.pt.pivot(r="island", c="species", v="body_mass_g", a="mean")

,island,Adelie,Chinstrap,Gentoo
0,Biscoe,3709.659091,NaN,5076.01626
1,Dream,3688.392857,3733.088235,NaN
2,Torgersen,3706.372549,NaN,NaN


## 2. Frequency Matrix via Row Counts (`a="n"`)

Count occurrences across dimensions (the direct successor to cross-tabulation):

In [3]:
penguins.pt.pivot(r="island", c="species", v="sex", a="n")

,island,Adelie,Chinstrap,Gentoo
0,Biscoe,44,0,124
1,Dream,56,68,0
2,Torgersen,52,0,0


## 3. Flexible Excel Layouts: Optional Rows or Columns

Just like dragging fields in Excel, `r` and `c` are completely optional.

In [4]:
# Omit c: vertical summary table by row group
penguins.pt.pivot(r="species", v="body_mass_g", a="mean")

,species,body_mass_g
0,Adelie,3700.662252
1,Chinstrap,3733.088235
2,Gentoo,5076.016260


In [5]:
# Omit r: horizontal summary table across column headers
penguins.pt.pivot(c="species", v="body_mass_g", a="mean")

,Adelie,Chinstrap,Gentoo
0,3700.662252,3733.088235,5076.01626


In [6]:
# Omit both r and c: grand scalar aggregation across the dataset
penguins.pt.pivot(v="body_mass_g", a="mean")

,body_mass_g
0,4201.754386


## 4. Multi-Level Dimensions & Multi-Metric Reporting

Pass a list of columns to `r`, `c`, or `v`:

In [7]:
# Hierarchical rows + multiple metric values
penguins.pt.pivot(
    r=["island", "sex"],
    c="species",
    v=["body_mass_g", "flipper_length_mm"],
    a="mean",
    fill_value=0
)


,island,sex,body_mass_g_Adelie,body_mass_g_Chinstrap,body_mass_g_Gentoo,flipper_length_mm_Adelie,flipper_length_mm_Chinstrap,flipper_length_mm_Gentoo
0,Biscoe,Female,3369.318182,0.000000,4679.741379,187.181818,0.000000,212.706897
1,Biscoe,Male,4050.000000,0.000000,5484.836066,190.409091,0.000000,221.540984
2,Biscoe,NaN,0.000000,0.000000,4587.500000,0.000000,0.000000,215.750000
3,Dream,Female,3344.444444,3527.205882,0.000000,187.851852,191.735294,0.000000
4,Dream,Male,4045.535714,3938.970588,0.000000,191.928571,199.911765,0.000000
5,Dream,NaN,2975.000000,0.000000,0.000000,179.000000,0.000000,0.000000
6,Torgersen,Female,3395.833333,0.000000,0.000000,188.291667,0.000000,0.000000
7,Torgersen,Male,4034.782609,0.000000,0.000000,194.913043,0.000000,0.000000
8,Torgersen,NaN,3681.250000,0.000000,0.000000,187.250000,0.000000,0.000000


## 5. Seamless Method Chaining & Plotting

Because `pt.pivot` always returns a clean, rectangular DataFrame with a flat RangeIndex, you can immediately chain into further transformations or `.pt.plot()`:

In [8]:
piv = penguins.pt.pivot(r="island", c="species", v="body_mass_g", a="mean", fill_value=0)
# Clean flat columns and standard index ready for downstream manipulation:
print("Columns:", list(piv.columns))
print("Index:", list(piv.index))
piv

Columns: ['island', 'Adelie', 'Chinstrap', 'Gentoo']
Index: [0, 1, 2]


,island,Adelie,Chinstrap,Gentoo
0,Biscoe,3709.659091,0.000000,5076.01626
1,Dream,3688.392857,3733.088235,0.00000
2,Torgersen,3706.372549,0.000000,0.00000
